# 10 - One shared U-Net: controlled strong-first curriculum

**T4 GPU; Run All.** Two fresh training runs, eight epochs each, with the original 487009-parameter U-Net and global L1. No GAN, expert routing, new loss or inference-time condition labels. Requires frozen notebooks 03/05/06/07 artifacts and restored images; no NBIS installation is needed. Earlier experiments/checkpoints are read-only.

Notebook 09 established learning on tiny fixed train sets, not generalization. Mixed balanced loss did not improve total training MAE over mixed global L1. We now test an optimization-order hypothesis, NOT a novel method or a proven explanation for M0's failure.

Strong-first curriculum (deliberately NOT easy-to-hard): source epochs 1-3 use severe dry_like/wet_like/generic; epochs 4-5 use moderate conditions; epochs 6-8 use all nine degradations plus identity. Conditions are sampled uniformly within each stage. The rationale is to test whether starting with stronger reconstruction signals helps before adding milder/identity cases. This may fail or harm original images; measure both.

Every task is a fixed (source image, source epoch) pair, defining its crop, simulator seed and condition through the unchanged notebook-07 dataset. Curriculum shuffles within source epochs. The control globally shuffles EXACTLY THE SAME tasks into eight equal-size epochs. Both see each task once, have identical batch sizes, update counts, initial weights, loss, optimizer and validation. Only task order differs. The global control's early epochs need not contain each source image once; equality holds over the full run.

Use all frozen M0 train sources and the SAME 72 validation sources/720 fixed validation patches. FP32 training with AMP/TF32 disabled, consistent with the successful notebook-09 diagnostic. Comparisons to old M0 also change precision and degradation dose; only curriculum versus its matched control isolates ordering. Eight epochs is a first controlled budget, not a guarantee of convergence.

Adam lr 0.0002, batch 8, 192x192 crops, no geometric augmentation. Select each run's best checkpoint by equal-condition mean validation L1, including the initial identity checkpoint. Report identity-image errors and post-8-bit changes separately. Validation has already been used for selection: it is NOT independent test evidence, a matching result, or a claim about real moisture labels. FVC2004 and test_synthetic stay sealed.

Save every completed epoch to Drive with hashes, optimizer/RNG state and content-addressed checkpoint names. Run All safely resumes even after a partially updated epoch; unfinished epochs repeat from the last durable checkpoint. Completed runs are verified before reuse. Do not edit the frozen experiment to resume; run only one copy at a time.

Outputs: MyDrive/FingerprintResearch/experiments/m1_shared_curriculum_v1_seed20261110/<configuration-hash>/{curriculum,shuffled_control}. Completion marker: SHARED CURRICULUM CONTROL COMPLETE. Send the final comparison and previews before starting recognition evaluation; lower patch MAE alone does not establish useful restoration.

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import inspect
import json
import math
import shutil
import sys
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Subset
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU kernel. A T4 is sufficient.')
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
M0_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'
SEED = 20261110
SETTINGS = {'phase_lengths': [3, 2, 3], 'batch_size': 8, 'patch_size': 192,
            'base_channels': 16, 'learning_rate': 0.0002, 'precision': 'FP32; AMP/TF32 disabled'}
RUN_NAMES = ('curriculum', 'shuffled_control')

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot load module: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module
print('GPU:', torch.cuda.get_device_name(0), '| Two matched runs,', sum(SETTINGS['phase_lengths']), 'epochs each.')

In [ ]:
split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))
for name, expected in split_protocol['file_sha256'].items():
    if file_digest(SPLIT_DIR / name) != expected:
        raise RuntimeError(f'Frozen split changed: {name}')
locked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)
m0_config = json.loads((M0_DIR / 'configuration.json').read_text(encoding='utf-8'))
sim_config = json.loads((SIM_DIR / 'configuration.json').read_text(encoding='utf-8'))
pair_config = json.loads((PAIR_DIR / 'configuration.json').read_text(encoding='utf-8'))
m0_best = json.loads((M0_DIR / 'best.json').read_text(encoding='utf-8'))
m0_completion = json.loads((M0_DIR / 'completion.json').read_text(encoding='utf-8'))
m0_sha = file_digest(M0_DIR / 'configuration.json')
if any(config['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json')
       for config in (m0_config, sim_config, pair_config)):
    raise RuntimeError('Split provenance differs across notebooks.')
if (m0_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or sim_config['normalization_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or m0_config['simulator_module_sha256'] != sim_config['module_sha256']
        or file_digest(SIM_DIR / 'fingerprint_degradation.py') != sim_config['module_sha256']):
    raise RuntimeError('Simulator/normalization provenance mismatch.')
for name, expected in m0_config['source_file_sha256'].items():
    if file_digest(M0_DIR / name) != expected:
        raise RuntimeError(f'Frozen M0 source changed: {name}')
if (m0_completion['status'] != 'm0_training_complete' or m0_completion['epochs_completed'] != m0_config['epochs']
        or m0_completion['configuration_sha256'] != m0_sha or m0_completion['best_checkpoint'] != m0_best
        or m0_best['configuration_sha256'] != m0_sha or file_digest(M0_DIR / m0_best['file']) != m0_best['sha256']):
    raise RuntimeError('M0 completion/checkpoint provenance mismatch.')
if (str(torch.__version__) != m0_config['torch_version'] or str(torch.version.cuda) != m0_config['torch_cuda']
        or Image.__version__ != m0_config['pillow_version'] or Image.__version__ != pair_config['pillow_version']
        or Image.__version__ != sim_config['pillow_version'] or np.__version__ != sim_config['numpy_version']):
    raise RuntimeError('Runtime library versions differ from the frozen experiment.')
if (sum(SETTINGS['phase_lengths']) != m0_config['epochs'] or any(length <= 0 for length in SETTINGS['phase_lengths'])
        or SETTINGS['patch_size'] != m0_config['patch_size'] or SETTINGS['base_channels'] != m0_config['base_channels']
        or SETTINGS['batch_size'] != m0_config['batch_size'] or SETTINGS['learning_rate'] != m0_config['learning_rate']):
    raise RuntimeError('Architecture, epoch budget, batch size and Adam learning rate must match M0.')
train_sources = pd.read_csv(M0_DIR / 'train_sources.csv', keep_default_na=False)
validation_sources = pd.read_csv(M0_DIR / 'validation_sources.csv', keep_default_na=False)
for split_name, frame in (('train', train_sources), ('validation', validation_sources)):
    if frame.empty or frame['relative_path'].duplicated().any():
        raise RuntimeError(f'Empty/duplicate {split_name} sources.')
    allowed = locked[locked['split'].eq(split_name)].set_index('relative_path')
    if set(frame['dataset']) != {'FVC2000', 'FVC2002'} or frame.groupby(['dataset', 'db']).ngroups != 6:
        raise RuntimeError('Expected six source databases only.')
    for record in frame.to_dict('records'):
        relative = record['relative_path']
        path = DATA_ROOT / relative
        if (relative not in allowed.index or allowed.loc[relative, 'sha256'] != record['sha256']
                or allowed.loc[relative, 'identity_key'] != record['identity_key']):
            raise RuntimeError(f'Wrong split/source identity: {relative}')
        if not path.is_file() or file_digest(path) != record['sha256']:
            raise RuntimeError(f'Image missing/changed: {path}. Restore with notebook 02.')
if (set(train_sources['identity_key']) & set(validation_sources['identity_key'])
        or set(train_sources['sha256']) & set(validation_sources['sha256'])):
    raise RuntimeError('Train/validation identity or byte overlap.')
if not validation_sources.groupby(['dataset', 'db']).size().eq(12).all():
    raise RuntimeError('Expected the unchanged 12 M0 validation sources per database.')
training = load_module('m1_original_model_v1', M0_DIR / 'restoration_baseline.py')
simulator = load_module('m1_original_simulator_v1', SIM_DIR / 'fingerprint_degradation.py')
print('Same M0 sources: train', len(train_sources), '| validation', len(validation_sources))

In [ ]:
DATA_SOURCE = '''"""Fixed M0 tasks; curriculum/control differ only in task order."""
from operator import index
from torch.utils.data import Dataset

class CurriculumPairs(Dataset):
    def __init__(self, base, records, root, simulator, dpi, seed, patch_size, epoch_conditions):
        self.sources = list(records)
        if not self.sources or not epoch_conditions:
            raise ValueError('Nonempty sources and epoch conditions are required.')
        self.datasets = []
        for source_epoch, conditions in enumerate(epoch_conditions):
            if not conditions:
                raise ValueError('Each source epoch must have conditions.')
            dataset = base.FingerprintPairs(self.sources, root, simulator, dpi, seed,
                                            patch_size=patch_size, training=True)
            dataset.epoch = source_epoch
            dataset.conditions = [tuple(condition) for condition in conditions]
            self.datasets.append(dataset)

    def __len__(self):
        return len(self.sources) * len(self.datasets)

    def __getitem__(self, task_id):
        task_id = index(task_id)
        if not 0 <= task_id < len(self):
            raise IndexError(f'Invalid task ID: {task_id}')
        source_epoch, image_index = divmod(task_id, len(self.sources))
        sample = self.datasets[source_epoch][image_index]
        return {**sample, 'task_id': task_id, 'source_epoch': source_epoch}
'''
module_path = LOCAL_ROOT / 'm1_curriculum_data.py'
module_path.parent.mkdir(parents=True, exist_ok=True)
module_path.write_text(DATA_SOURCE, encoding='utf-8')
curriculum_data = load_module('m1_curriculum_data_v1', module_path)
all_conditions = [('identity', 'none')] + [(family, severity)
    for family in ('dry_like', 'wet_like', 'generic') for severity in ('mild', 'moderate', 'severe')]
stage_conditions = [[(family, severity) for family in ('dry_like', 'wet_like', 'generic')]
                    for severity in ('severe', 'moderate')] + [all_conditions]
epoch_conditions = [conditions for conditions, count in zip(stage_conditions, SETTINGS['phase_lengths'])
                    for _ in range(count)]
EPOCHS = len(epoch_conditions)
SOURCE_COUNT = len(train_sources)
task_dataset = curriculum_data.CurriculumPairs(training, train_sources.to_dict('records'), DATA_ROOT,
    simulator, pair_config['source_dpi'], SEED, SETTINGS['patch_size'], epoch_conditions)
orders = {'curriculum': np.concatenate([epoch * SOURCE_COUNT + np.random.default_rng(SEED + epoch).permutation(SOURCE_COUNT)
                                     for epoch in range(EPOCHS)]),
          'shuffled_control': np.random.default_rng(SEED + 10000).permutation(len(task_dataset))}
for name, order in orders.items():
    if not np.array_equal(np.sort(order), np.arange(len(task_dataset))):
        raise RuntimeError(f'Run does not contain every fixed task exactly once: {name}')
validation_dataset = training.FingerprintPairs(validation_sources.to_dict('records'), DATA_ROOT, simulator,
    pair_config['source_dpi'], SEED, patch_size=SETTINGS['patch_size'], training=False)
validation_loader = DataLoader(validation_dataset, batch_size=SETTINGS['batch_size'], shuffle=False,
    num_workers=0, pin_memory=DEVICE.type == 'cuda', generator=torch.Generator().manual_seed(SEED))
print('Tasks per run:', len(task_dataset), '| Updates per run:', EPOCHS * math.ceil(SOURCE_COUNT / SETTINGS['batch_size']))
print('Both runs contain exactly the same task IDs; fixed validation patches:', len(validation_dataset))

In [ ]:
def new_model():
    torch.manual_seed(SEED)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(SEED)
    return training.RestorationUNet(base=SETTINGS['base_channels']).to(DEVICE)

@torch.inference_mode()
def evaluate(model):
    model.eval()
    totals = {}
    for batch in validation_loader:
        xs, ys = batch['input'].to(DEVICE), batch['target'].to(DEVICE)
        predictions = model(xs).float()
        if predictions.shape != ys.shape or not torch.isfinite(predictions).all() or predictions.min() < 0 or predictions.max() > 1:
            raise RuntimeError('Validation output shape/range/finiteness failure.')
        png_before, png_after, png_target = torch.round(xs * 255), torch.round(predictions * 255), torch.round(ys * 255)
        columns = {'mae': (predictions - ys).abs().mean((1, 2, 3)),
            'mse': (predictions - ys).square().mean((1, 2, 3)), 'input_mae': (xs - ys).abs().mean((1, 2, 3)),
            'png_mae': (png_after - png_target).abs().mean((1, 2, 3)) / 255,
            'png_changed_fraction': png_after.ne(png_before).float().mean((1, 2, 3)),
            'png_identical_image_fraction': png_after.eq(png_before).flatten(1).all(1).float()}
        values = {name: tensor.cpu().tolist() for name, tensor in columns.items()}
        for index, condition in enumerate(batch['condition']):
            total = totals.setdefault(condition, {'samples': 0, **{name: 0.0 for name in columns}})
            total['samples'] += 1
            for name in columns:
                total[name] += values[name][index]
    expected = {f'{family}/{severity}' for family, severity in all_conditions}
    if set(totals) != expected or any(row['samples'] != len(validation_sources) for row in totals.values()):
        raise RuntimeError('Validation must cover every frozen source in all ten conditions.')
    rows = [{'condition': condition, 'samples': total['samples'],
             **{name: value / total['samples'] for name, value in total.items() if name != 'samples'}}
            for condition, total in sorted(totals.items())]
    count = sum(row['samples'] for row in rows)
    result = {name: sum(row[name] * row['samples'] for row in rows) / count
              for name in ('mae', 'mse', 'input_mae', 'png_mae', 'png_changed_fraction', 'png_identical_image_fraction')}
    result.update(identity_mae=next(row['mae'] for row in rows if row['condition'] == 'identity/none'), conditions=rows)
    return result

def save_checkpoint(directory, payload, is_best):
    completed = payload['completed_epochs']
    local = LOCAL_EXPERIMENT / f'{directory.name}_epoch_{completed:03d}.pt'
    torch.save(payload, local)
    expected = file_digest(local)
    name = f'epoch_{completed:03d}_{expected[:16]}.pt'
    destination = directory / name
    if destination.exists() and file_digest(destination) != expected:
        raise RuntimeError(f'Checkpoint filename collision: {destination}')
    if not destination.exists():
        pending = destination.with_suffix('.pt.part')
        shutil.copyfile(local, pending)
        if file_digest(pending) != expected:
            raise RuntimeError(f'Checkpoint Drive copy checksum failure: {directory.name}')
        pending.replace(destination)
    pointer = {'file': name, 'sha256': expected, 'configuration_sha256': experiment_sha,
               'completed_epochs': completed, 'validation_mae': payload['last_validation']['mae']}
    if is_best:
        write_verified(directory / 'best.json', json_bytes(pointer))
    write_verified(directory / 'latest.json', json_bytes(pointer))
    local.unlink()

def load_checkpoint(directory, pointer_name):
    pointer = json.loads((directory / pointer_name).read_text(encoding='utf-8'))
    path = directory / pointer['file']
    if pointer['configuration_sha256'] != experiment_sha or file_digest(path) != pointer['sha256']:
        raise RuntimeError(f'Checkpoint pointer/integrity mismatch: {path}')
    state = torch.load(path, map_location='cpu', weights_only=True)
    if (state['configuration_sha256'] != experiment_sha or state['run'] != directory.name
            or state['completed_epochs'] != pointer['completed_epochs']
            or len(state['history']) != state['completed_epochs']):
        raise RuntimeError(f'Checkpoint payload mismatch: {path}')
    return pointer, state

def load_completed_run(directory):
    marker = directory / 'completion.json'
    if not marker.is_file():
        return None
    saved = json.loads(marker.read_text(encoding='utf-8'))
    if (saved['configuration_sha256'] != experiment_sha or saved['run'] != directory.name
            or saved['epochs_completed'] != EPOCHS):
        raise RuntimeError(f'Completed experiment provenance mismatch: {directory}')
    required = {'initial_validation.json', 'history.csv', 'best_validation.json', 'best_validation_conditions.csv',
                'best_validation_preview.png', 'best.json', 'latest.json', saved['result']['best_checkpoint']['file']}
    if not required.issubset(saved['file_sha256']):
        raise RuntimeError(f'Completed experiment is missing required reports: {directory}')
    for name, expected in saved['file_sha256'].items():
        if file_digest(directory / name) != expected:
            raise RuntimeError(f'Completed experiment integrity failure: {directory / name}')
    latest, _ = load_checkpoint(directory, 'latest.json')
    if latest['completed_epochs'] != EPOCHS:
        raise RuntimeError(f'Completed experiment latest checkpoint is incomplete: {directory}')
    return saved['result']

def train_run(name):
    directory = REPORT_DIR / name
    directory.mkdir(parents=True, exist_ok=True)
    finished = load_completed_run(directory)
    if finished is not None:
        print('Resumed completed experiment:', name, flush=True)
        return finished
    model = new_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=SETTINGS['learning_rate'])
    def payload(completed_epochs, history, best_mae, validation):
        return {'configuration_sha256': experiment_sha, 'run': name, 'completed_epochs': completed_epochs,
            'history': history, 'best_mae': best_mae, 'last_validation': validation,
            'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
            'torch_rng': torch.get_rng_state(), 'cuda_rng': torch.cuda.get_rng_state_all()}
    if (directory / 'latest.json').is_file():
        latest, state = load_checkpoint(directory, 'latest.json')
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        torch.set_rng_state(state['torch_rng'])
        torch.cuda.set_rng_state_all(state['cuda_rng'])
        start_epoch, history, best_mae = state['completed_epochs'], state['history'], state['best_mae']
        print(f'{name}: resuming {start_epoch}/{EPOCHS} completed epochs', flush=True)
    else:
        initial = evaluate(model)
        if abs(initial['mae'] - initial['input_mae']) > 1e-7 or initial['identity_mae'] > 1e-7:
            raise RuntimeError('Fresh model did not start as identity.')
        start_epoch, history, best_mae = 0, [], initial['mae']
        write_verified(directory / 'initial_validation.json', json_bytes(initial))
        save_checkpoint(directory, payload(0, history, best_mae, initial), True)
    if not 0 <= start_epoch <= EPOCHS:
        raise RuntimeError('Checkpoint completed-epoch count is outside the configured budget.')
    for epoch in range(start_epoch, EPOCHS):
        started = time.monotonic()
        epoch_tasks = orders[name][epoch * SOURCE_COUNT:(epoch + 1) * SOURCE_COUNT].tolist()
        loader = DataLoader(Subset(task_dataset, epoch_tasks), batch_size=SETTINGS['batch_size'], shuffle=False,
            num_workers=0, pin_memory=DEVICE.type == 'cuda', generator=torch.Generator().manual_seed(SEED + epoch))
        model.train()
        loss_sum, seen, consumed = 0.0, 0, []
        for step, batch in enumerate(loader, start=1):
            xs, ys = batch['input'].to(DEVICE), batch['target'].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            predictions = model(xs)
            loss = (predictions - ys).abs().mean()
            if not torch.isfinite(loss):
                raise RuntimeError(f'Non-finite training loss: {name}, epoch {epoch + 1}, step {step}')
            loss.backward()
            optimizer.step()
            loss_sum += float(loss.detach()) * len(xs)
            seen += len(xs)
            consumed.extend(batch['task_id'].tolist())
            if step % 100 == 0:
                print(f'{name}: epoch {epoch + 1}/{EPOCHS}, step {step}/{len(loader)}, L1={loss_sum / seen:.5f}', flush=True)
        if seen != SOURCE_COUNT or consumed != epoch_tasks:
            raise RuntimeError('Epoch did not consume the exact frozen task order.')
        validation = evaluate(model)
        is_best = validation['mae'] < best_mae
        best_mae = min(best_mae, validation['mae'])
        counts = np.bincount(np.asarray(consumed) // SOURCE_COUNT, minlength=EPOCHS)
        history.append({'epoch': epoch + 1, 'train_mae': loss_sum / seen, 'validation_mae': validation['mae'],
            'validation_identity_mae': validation['identity_mae'], 'seconds': time.monotonic() - started,
            **{f'source_epoch_{index + 1}_tasks': int(count) for index, count in enumerate(counts)}})
        save_checkpoint(directory, payload(epoch + 1, history, best_mae, validation), is_best)
        write_csv(directory / 'history.csv', pd.DataFrame(history))
        print(f'{name}: epoch {epoch + 1} saved, val={validation["mae"]:.5f}, identity={validation["identity_mae"]:.5f}, best={is_best}', flush=True)
    best_pointer, best_state = load_checkpoint(directory, 'best.json')
    model.load_state_dict(best_state['model'])
    best_validation = evaluate(model)
    if abs(best_validation['mae'] - best_pointer['validation_mae']) > 1e-7:
        raise RuntimeError('Best-checkpoint validation recomputation differs from the saved result.')
    write_csv(directory / 'history.csv', pd.DataFrame(history))
    write_verified(directory / 'best_validation.json', json_bytes(best_validation))
    write_csv(directory / 'best_validation_conditions.csv', pd.DataFrame(best_validation['conditions']))
    figure, axes = plt.subplots(10, 3, figsize=(9, 24))
    with torch.inference_mode():
        for row in range(10):
            sample = validation_dataset[row]
            output = model(sample['input'][None].to(DEVICE)).cpu()[0, 0].numpy()
            images = (sample['input'][0].numpy(), output, sample['target'][0].numpy())
            for column, (title, image) in enumerate(zip(('input', name, 'reference'), images)):
                axes[row, column].imshow(np.rint(image * 255).astype(np.uint8), cmap='gray', vmin=0, vmax=255)
                axes[row, column].set_title(f'{sample["condition"]} | {title}', fontsize=7)
                axes[row, column].axis('off')
    figure.tight_layout()
    figure.savefig(directory / 'best_validation_preview.png', dpi=140)
    plt.show()
    plt.close(figure)
    result = {'run': name, 'best_epoch': best_pointer['completed_epochs'], 'epochs_completed': EPOCHS,
        'best_checkpoint': best_pointer, 'validation': best_validation,
        'recognition_improvement_verified': False, 'external_test_evaluated': False}
    latest = json.loads((directory / 'latest.json').read_text(encoding='utf-8'))
    files = {'initial_validation.json', 'history.csv', 'best_validation.json', 'best_validation_conditions.csv',
             'best_validation_preview.png', 'best.json', 'latest.json', best_pointer['file'], latest['file']}
    marker = {'configuration_sha256': experiment_sha, 'run': name, 'epochs_completed': EPOCHS,
              'file_sha256': {filename: file_digest(directory / filename) for filename in sorted(files)}, 'result': result}
    write_verified(directory / 'completion.json', json_bytes(marker))
    return result

In [ ]:
helper_source = '\n\n'.join(inspect.getsource(function) for function in
    (new_model, evaluate, save_checkpoint, load_checkpoint, load_completed_run, train_run))
source_files = {name: (M0_DIR / name).read_bytes() for name in
                ('restoration_baseline.py', 'train_sources.csv', 'validation_sources.csv')}
source_files.update({'curriculum_data.py': DATA_SOURCE.encode('utf-8'),
                     'training_helpers.py.txt': helper_source.encode('utf-8')})
for name, order in orders.items():
    source_files[f'order_{name}.csv'] = pd.DataFrame({'task_id': order}).to_csv(index=False, lineterminator='\n').encode()
experiment_config = {'version': 'm1_shared_curriculum_v1', 'seed': SEED, 'settings': SETTINGS,
    'm0_configuration_sha256': m0_sha, 'm0_best_checkpoint': m0_best,
    'simulator_configuration_sha256': file_digest(SIM_DIR / 'configuration.json'),
    'epoch_conditions': epoch_conditions, 'run_names': RUN_NAMES,
    'task_definition': 'fixed original source epoch/image pair; unchanged M0 crop and simulator seed rules',
    'control': 'global permutation of exactly the same tasks; only ordering differs',
    'selection_metric': 'mean fixed validation L1; initial identity checkpoint included',
    'loss': 'global L1', 'optimizer': 'Adam', 'num_workers': 0,
    'tasks_per_run': len(task_dataset), 'updates_per_run': EPOCHS * math.ceil(SOURCE_COUNT / SETTINGS['batch_size']),
    'source_file_sha256': {name: hashlib.sha256(data).hexdigest() for name, data in source_files.items()},
    'torch_version': str(torch.__version__), 'torch_cuda': str(torch.version.cuda),
    'numpy_version': np.__version__, 'pillow_version': Image.__version__,
    'gpu_name': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None,
    'gpu_compute_capability': list(torch.cuda.get_device_capability(0)) if DEVICE.type == 'cuda' else None,
    'validation_is_not_independent_test': True}
config_data = json_bytes(experiment_config)
experiment_sha = hashlib.sha256(config_data).hexdigest()
REPORT_DIR = PROJECT_ROOT / 'experiments' / 'm1_shared_curriculum_v1_seed20261110' / experiment_sha[:20]
LOCAL_EXPERIMENT = LOCAL_ROOT / 'm1_training' / experiment_sha[:20]
for directory in (REPORT_DIR, LOCAL_EXPERIMENT):
    directory.mkdir(parents=True, exist_ok=True)
for name, data in {**source_files, 'configuration.json': config_data}.items():
    path = REPORT_DIR / name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Frozen controlled experiment changed: {path}')
    if not path.exists():
        write_verified(path, data)
print('Controlled experiment frozen:', experiment_sha)
print('Drive outputs:', REPORT_DIR)

In [ ]:
results = [train_run(name) for name in RUN_NAMES]
print('Both matched-budget training runs completed and persisted.')

In [ ]:
m0_model = new_model()
m0_state = torch.load(M0_DIR / m0_best['file'], map_location='cpu', weights_only=True)
if m0_state['configuration_sha256'] != m0_sha or m0_state['epoch'] != m0_best['epoch']:
    raise RuntimeError('M0 reference checkpoint payload mismatch.')
m0_model.load_state_dict(m0_state['model'])
m0_validation = evaluate(m0_model)
write_verified(REPORT_DIR / 'm0_reference_validation.json', json_bytes(m0_validation))
comparison_rows = [{'method': 'input', 'best_epoch': 0, 'validation_mae': m0_validation['input_mae'],
                    'relative_mae_reduction': 0.0, 'identity_mae': 0.0, 'png_changed_fraction': 0.0}]
for name, epoch, validation in [('m0_reference', m0_best['epoch'] + 1, m0_validation)] + [
    (result['run'], result['best_epoch'], result['validation']) for result in results]:
    comparison_rows.append({'method': name, 'best_epoch': epoch, 'validation_mae': validation['mae'],
        'relative_mae_reduction': (validation['input_mae'] - validation['mae']) / validation['input_mae'],
        'identity_mae': validation['identity_mae'], 'png_changed_fraction': validation['png_changed_fraction']})
comparison = pd.DataFrame(comparison_rows)
write_csv(REPORT_DIR / 'experiment_comparison.csv', comparison)
condition_frames = [pd.DataFrame(m0_validation['conditions']).assign(method='m0_reference')]
condition_frames.extend(pd.DataFrame(result['validation']['conditions']).assign(method=result['run']) for result in results)
write_csv(REPORT_DIR / 'condition_comparison.csv', pd.concat(condition_frames, ignore_index=True))
print(comparison.to_string(index=False))
print('Relative reduction is a fraction; identity MAE and PNG changes must be considered separately.')
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for name in RUN_NAMES:
    history = pd.read_csv(REPORT_DIR / name / 'history.csv')
    axes[0].plot(history['epoch'], history['validation_mae'], marker='o', label=name)
    axes[1].plot(history['epoch'], history['validation_identity_mae'], marker='o', label=name)
axes[0].axhline(m0_validation['input_mae'], color='gray', linestyle='--', label='input')
axes[0].set_title('Fixed validation L1')
axes[1].set_title('Original-crop identity MAE')
for axis in axes:
    axis.set_xlabel('Completed training epoch')
    axis.legend(fontsize=8)
figure.tight_layout()
figure.savefig(REPORT_DIR / 'controlled_learning_curves.png', dpi=140)
plt.show()
plt.close(figure)
summary_files = ['configuration.json', *source_files, 'm0_reference_validation.json',
                 'experiment_comparison.csv', 'condition_comparison.csv', 'controlled_learning_curves.png']
completion = {'status': 'shared_curriculum_control_complete', 'configuration_sha256': experiment_sha,
    'file_sha256': {name: file_digest(REPORT_DIR / name) for name in summary_files},
    'run_completion_sha256': {name: file_digest(REPORT_DIR / name / 'completion.json') for name in RUN_NAMES},
    'recognition_improvement_verified': False, 'external_test_evaluated': False,
    'interpretation': 'exploratory validation; only matched runs isolate ordering; neither fit nor MAE proves recognition gains'}
write_verified(REPORT_DIR / 'completion.json', json_bytes(completion))
print('SHARED CURRICULUM CONTROL COMPLETE:', REPORT_DIR)
print('Send experiment_comparison, condition_comparison and previews before the matching decision.')